# Model Training Pipeline

This notebook focuses on training the gesture recognition model. It loads and preprocesses the dataset, selects the appropriate hand(s) based on configuration, and reshapes the sequences for model input. The model architecture is defined dynamically from a configuration file, allowing flexible use of GRU, Dense, and Dropout layers. After training, the trained model is saved for later use in inference or evaluation.

In [1]:
import numpy as np
import json
from typing import Tuple
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


def load_config(config_path: str):
    """
    Load and parse the configuration file.

    Args:
        config_path: Path to the JSON configuration file.

    Returns:
        Parsed configuration dictionary.
    """
    with open(config_path, 'r') as f:
        return json.load(f)


def load_dataset(dataset_path: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load dataset from compressed file.

    Args:
        dataset_path: Path to the .npz dataset file.

    Returns:
        Tuple containing feature array X and label array y.
    """
    data = np.load(dataset_path)
    return data['X'], data['y']


def select_hands(X: np.ndarray, hand_selection: str) -> np.ndarray:
    """
    Select specific hand data based on configuration.

    Args:
        X: Input data of shape (samples, sequence_length, landmarks, coords).
        hand_selection: 'left', 'right', or 'both'.

    Returns:
        Filtered dataset based on selected hands.
    """
    if hand_selection == 'left':
        X = X[:, :, 22:, :]
        print(f"Using left hand only: {X.shape}")
    elif hand_selection == 'right':
        X = X[:, :, :22, :]
        print(f"Using right hand only: {X.shape}")
    else:
        print(f"Using both hands: {X.shape}")

    return X


def preprocess_data(X: np.ndarray, y_labels: np.ndarray, sequence_length: int, num_classes: int):
    """
    Reshape input data and convert labels to one-hot encoding.

    Args:
        X: Input feature array.
        y_labels: Label array.
        sequence_length: Length of each sequence.
        num_classes: Number of output classes.

    Returns:
        Processed feature array and one-hot encoded labels.
    """
    num_samples = X.shape[0]
    X = X.reshape(num_samples, sequence_length, -1)
    y = to_categorical(y_labels, num_classes=num_classes)
    return X, y


def split_data(X, y, y_labels, test_size: float, stratify: bool):
    """
    Split dataset into training and testing sets.

    Args:
        X: Features.
        y: One-hot labels.
        y_labels: Original labels.
        test_size: Proportion of test data.
        stratify: Whether to stratify split.

    Returns:
        Split datasets: X_train, X_test, y_train, y_test.
    """
    stratify_param = y_labels if stratify else None

    return train_test_split(
        X, y,
        test_size=test_size,
        stratify=stratify_param,
        random_state=42
    )


def build_model(sequence_length: int, feature_dim: int, layers_config, num_classes: int):
    """
    Build a Sequential model based on configuration.

    Args:
        sequence_length: Length of input sequences.
        feature_dim: Number of features per timestep.
        layers_config: List of layer configuration dictionaries.
        num_classes: Number of output classes.

    Returns:
        Compiled Keras model.
    """
    model = Sequential()

    for i, layer_config in enumerate(layers_config):
        layer_type = layer_config['type']

        if layer_type == 'TimeDistributed_Dense':
            units = layer_config['units']
            activation = layer_config['activation']

            if i == 0:
                model.add(TimeDistributed(
                    Dense(units, activation=activation),
                    input_shape=(sequence_length, feature_dim)
                ))
            else:
                model.add(TimeDistributed(Dense(units, activation=activation)))

        elif layer_type == 'Dropout':
            model.add(Dropout(layer_config['rate']))

        elif layer_type == 'GRU':
            model.add(GRU(
                layer_config['units'],
                return_sequences=layer_config.get('return_sequences', False)
            ))

        elif layer_type == 'Dense':
            units = layer_config.get('units', num_classes)
            model.add(Dense(units, activation=layer_config['activation']))

    return model


def compile_model(model, optimizer, loss, metrics):
    """
    Compile the Keras model.

    Args:
        model: Keras model.
        optimizer: Optimizer name or object.
        loss: Loss function.
        metrics: List of metrics.
    """
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )


def train_model(model, X_train, y_train, epochs, batch_size, validation_split):
    """
    Train the model.

    Returns:
        Training history object.
    """
    return model.fit(
        X_train,
        y_train,
        epochs=epochs,
        validation_split=validation_split,
        batch_size=batch_size,
        verbose=1
    )


def evaluate_model(model_path: str, X_test, y_test):
    """
    Evaluate model performance on test set.

    Args:
        model_path: Path to saved model.
        X_test: Test features.
        y_test: Test labels.

    Returns:
        Accuracy and confusion matrix.
    """
    model = load_model(model_path)

    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, cm


def main():
    config = load_config('../config.json')

    common_config = config['common']
    ACTIONS = common_config['actions']
    SEQUENCE_LENGTH = common_config['sequence_length']

    train_config = config['train_model']
    DATASET_PATH = train_config['dataset_path']
    MODEL_EXPORT_NAME = train_config['model_export_name']
    HAND_SELECTION = train_config['hand_selection']

    model_arch = train_config['model_architecture']
    LAYERS_CONFIG = model_arch['layers']

    training_config = train_config['training']
    EPOCHS = training_config['epochs']
    BATCH_SIZE = training_config['batch_size']
    VALIDATION_SPLIT = training_config['validation_split']
    OPTIMIZER = training_config['optimizer']
    LOSS = training_config['loss']
    METRICS = training_config['metrics']

    split_config = train_config['data_split']
    TEST_SIZE = split_config['test_size']
    STRATIFY = split_config['stratify']

    X, y_labels = load_dataset(DATASET_PATH)
    X = select_hands(X, HAND_SELECTION)

    X, y = preprocess_data(X, y_labels, SEQUENCE_LENGTH, len(ACTIONS))

    X_train, X_test, y_train, y_test = split_data(
        X, y, y_labels, TEST_SIZE, STRATIFY
    )

    print("X_train:", X_train.shape, "X_test:", X_test.shape)
    print("y_train:", y_train.shape, "y_test:", y_test.shape)

    model = build_model(SEQUENCE_LENGTH, X.shape[2], LAYERS_CONFIG, len(ACTIONS))
    compile_model(model, OPTIMIZER, LOSS, METRICS)

    model.summary()

    train_model(
        model,
        X_train,
        y_train,
        EPOCHS,
        BATCH_SIZE,
        VALIDATION_SPLIT
    )

    model.save(MODEL_EXPORT_NAME)
    print(f"Model saved to {MODEL_EXPORT_NAME}")

    acc, cm = evaluate_model(MODEL_EXPORT_NAME, X_test, y_test)

    print(f"Test accuracy: {acc:.4f}")
    print("Confusion matrix:")
    print(cm)


if __name__ == "__main__":
    main()

2026-04-04 14:57:39.297468: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-04 14:57:39.351405: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-04 14:57:40.699708: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Using right hand only: (3310, 20, 22, 3)
X_train: (2813, 20, 66) X_test: (497, 20, 66)
y_train: (2813, 12) y_test: (497, 12)


/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-04-04 14:57:42.399305: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 64)         │         4,288 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12)             │           396 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,684 (221.42 KB)

 Trainable params: 56,684 (221.42 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.5951 - loss: 1.2158 - val_accuracy: 0.9520 - val_loss: 0.2753
Epoch 2/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9676 - loss: 0.1616 - val_accuracy: 0.9982 - val_loss: 0.0300
Epoch 3/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9916 - loss: 0.0391 - val_accuracy: 0.9982 - val_loss: 0.0123
Epoch 4/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9933 - loss: 0.0272 - val_accuracy: 0.8881 - val_loss: 0.2261
Epoch 5/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9858 - loss: 0.0567 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 6/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 1.0000 - loss: 0.0033 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 7/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 8/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accu

Model saved to hand_gesture_model.h5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
Test accuracy: 1.0000
Confusion matrix:
[[38  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 38  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 38  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 39  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 38  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 38  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 39  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 39  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 38  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 38  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 38  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 76]]
